In [3]:
import pandas as pd
import json
import boto3
import vaex

# this section goes into a lambda function
s3 = boto3.Session(profile_name='personal').client('s3')


def handler(event):
    body = json.loads(event['body'])
    idf, words2id, jobs = get_files()
    words = pd.read_parquet('./seek/words-sm.parquet')
    for key in body:
        jobs = jobs[jobs[key] == body[key]]
    ids = jobs['id']
    id_filter = ids.to_frame().set_index('id')
    filtered_words = words.merge(id_filter, on='id', how='inner').drop(columns='id')
    freq = filtered_words.groupby('word_id').agg({'count': 'sum'})
    freq = freq.merge(words2id, on='word_id')
    freq = freq.set_index('word')['count']
    idf = idf.set_index('word')['count']
    df_idf = (freq * idf).dropna()
    body = (df_idf
        .sort_values(ascending=False)[:50]
        .to_frame()
        .reset_index()
        .to_numpy()
        .tolist())
    return {
        'headers': {
            'Access-Control-Allow-Origin': '*',
        },
        'statusCode': 200, 
        'body': json.dumps(body)
    }



def get_files():
    filenames = ['idf.parquet', 'words2id.parquet', 'jobs.parquet']
    return [download(name) for name in filenames]


def download(name):
    s3.download_file('jobs--001', name, f'./{name}')
    return pd.read_parquet(f'./{name}')
    

handler({'body': json.dumps({'sector': 'Information & Communication Technology'})})

KeyboardInterrupt: 

In [ ]:
raw = \
'''Host: hidemy.name
User-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:105.0) Gecko/20100101 Firefox/105.0
Accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8
Accept-Language: en-US,en;q=0.5
Accept-Encoding: gzip, deflate, br
DNT: 1
Connection: keep-alive
Upgrade-Insecure-Requests: 1
Sec-Fetch-Dest: document
Sec-Fetch-Mode: navigate
Sec-Fetch-Site: none
Sec-Fetch-User: ?1
Pragma: no-cache
Cache-Control: no-cache'''

proxy_headers = {}
for line in raw.split('\n'):
    key, value = line.split(': ')
    proxy_headers[key] = value

import pandas as pd
import requests

text = requests.get('https://hidemy.name/en/proxy-list/', headers=proxy_headers).text
pd.read_html(text)

[        IP address   Port             Country, City    Speed    Type  \
 0    66.42.224.229  41679  United States Cincinnati  5100 ms  SOCKS4   
 1    184.178.172.3   4145     United States Choctaw  4860 ms  SOCKS5   
 2   72.206.181.123   4145             United States  4160 ms  SOCKS5   
 3      98.162.25.4  31654             United States  2780 ms  SOCKS5   
 4    49.156.38.126   5678       Cambodia Phnom Penh  3360 ms  SOCKS4   
 ..             ...    ...                       ...      ...     ...   
 59    45.8.104.172     80                   Curacao   420 ms    HTTP   
 60   203.23.106.96     80                    Cyprus   320 ms    HTTP   
 61   203.24.103.75     80   Virgin Islands, British    20 ms    HTTP   
 62    45.8.105.167     80                   Curacao   220 ms    HTTP   
 63     45.12.30.17     80                   Romania   240 ms    HTTP   
 
    Anonymity Latest update  
 0       High     1 minutes  
 1       High     2 minutes  
 2       High     2 minutes  
 3